In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()

openai_client = OpenAI(api_key = os.getenv("OPENAI_KEY"))

In [2]:
from utils.postgresql import db

raw_business_df = db.query("""
SELECT
    "id",
    "name",
    "address",
    "city",
    'CA' AS "state",
    "zip",
    "latitude",
    "longitude",
    "avg_rating" as "average_rating",
    "categories",
    "url"
FROM ca_business
WHERE city = 'San Diego'
""")

raw_business_df.head()

,id,name,address,city,state,zip,latitude,longitude,average_rating,categories,url
0,54839,Island Prime,"Island Prime, 880 Harbor Island Dr, San Diego,...",San Diego,CA,92101,32.724112,-117.188648,4.6,"[Steak house, Bar, Fine dining restaurant]",https://www.google.com/maps/place//data=!4m2!3...
1,54841,"Devine Pastabilities, Torpasta of San Diego","Devine Pastabilities, Torpasta of San Diego, 3...",San Diego,CA,92110,32.750897,-117.214233,4.6,"[Italian restaurant, Restaurant]",https://www.google.com/maps/place//data=!4m2!3...
2,54855,Serenity Shop,"Serenity Shop, 4740 Clairemont Mesa Blvd, San ...",San Diego,CA,92117,32.834960,-117.188829,4.7,"[Gift shop, Book store, Greeting card shop, Mu...",https://www.google.com/maps/place//data=!4m2!3...
3,54869,Prepkitchen Little Italy,"Prepkitchen Little Italy, 1660 India St, San D...",San Diego,CA,92101,32.722769,-117.168522,4.4,"[American restaurant, Bar, Mediterranean resta...",https://www.google.com/maps/place//data=!4m2!3...
4,54873,Citibank ATM,"Citibank ATM, 3295 Palm Ave, San Diego, CA 92154",San Diego,CA,92154,32.583403,-117.062444,3.0,[ATM],https://www.google.com/maps/place//data=!4m2!3...


In [3]:
with open("data/raw_business.json", "w") as f:
    json.dump(raw_business_df.to_dict(orient="records"), f)

In [10]:
def llm_clean_business(batch):
    system = """
You are a data cleaning assistant responsible for standardizing business records.
You must follow ALL rules exactly and return ONLY valid JSON.

===========================
### OUTPUT SCHEMA
For EACH business record, output an object matching this exact JSON structure:

{
  "id": "",
  "name": "",
  "street": "",
  "city": "",
  "state": "",
  "zip": "",
  "main_category": "",
  "sub_category": ""
}

- Do NOT add extra fields.
- Do NOT remove any fields.
- All values must be strings (empty string if unknown).
- Return an array of cleaned objects.

===========================
### TASK REQUIREMENTS

#### 1. BUSINESS NAME STANDARDIZATION
- Correct misspellings (e.g., "Starbuck Corop" → "Starbucks").
- Remove store numbers or location numbers (e.g., "#10234", "#12", "Store 55").
- Remove extra descriptors (e.g., "LLC", "Inc", “Co.” unless essential).
- Keep only the canonical brand/business name.

Examples:
Input: "Starbucks #10234" → Output: "Starbucks"
Input: "McDonals Resaurant" → Output: "McDonald's"
Input: "7 Eleven Store 88" → Output: "7-Eleven"

#### 2. ADDRESS EXTRACTION & NORMALIZATION
Given an address string OR separate fields, extract:
- street (include number + street name, standardized abbreviations: St, Rd, Ave)
- city
- state (2-letter code)
- zip (5 digits)

Fix misspellings and normalize:
- "San Diago" → "San Diego"
- "Californa" → "CA"
- "Covoy Stret" → "Convoy St"

If an address is incomplete, return best cleaned version and leave missing fields as "".

Example:
Input: "Starbucks 12042 Covoy Street, San Diago CA 92111"
Output:
  street: "12042 Convoy St"
  city: "San Diego"
  state: "CA"
  zip: "92111"

#### 3. BUSINESS CATEGORIZATION
Use only the standardized category vocabulary below.

### MAIN_CATEGORIES:
- Food & Drink
- Retail
- Professional Services
- Healthcare & Medical
- Finance & Insurance
- Real Estate
- Education & Childcare
- Government & Public Services
- Arts, Entertainment & Recreation
- Hospitality & Lodging
- Transportation & Logistics
- Automotive Services
- Personal Services
- Home Services & Contractors
- Technology & Telecommunications
- Manufacturing
- Construction
- Agriculture & Farming
- Energy & Utilities
- Media & Communications
- Legal Services
- Nonprofit & Community Organizations
- Religious Organizations
- Waste Management & Environmental Services

### SUB_CATEGORY EXAMPLES:
Food & Drink → "Coffee Shop", "Chinese Restaurant", "Bakery", "Bar", "Fast Food"
Retail → "Convenience Store", "Clothing Store", "Grocery Store"
Healthcare → "Dentist", "Urgent Care", "Optometrist"
Automotive → "Auto Repair", "Gas Station"
Personal Services → "Hair Salon", "Nail Salon", "Spa"

Choose the most specific correct sub_category.

Example:
Input: "Starbucks"
Output:
  main_category: "Food & Drink"
  sub_category: "Coffee Shop"

Example:
Input: "Shell Gas"
Output:
  main_category: "Automotive Services"
  sub_category: "Gas Station"

===========================
### OUTPUT VALIDATION RULES
- Output MUST be valid JSON.
- Output MUST be an array.
- Do NOT include explanations or comments.
- Do NOT wrap JSON in code blocks.
- If uncertain, make the best reasonable guess without inventing nonexistent information.

===========================
### BEGIN PROCESSING THE USER DATA
"""

    user = f"Clean the following business records:\n{json.dumps(batch, indent=2)}"

    response = openai_client.chat.completions.create(
        model="gpt-4.1",
        temperature=0,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )

    return json.loads(response.choices[0].message.content)

In [11]:
# Delete LLM business json file if it exists
CLEAN_BUSINESS_JSON = "data/clean_business.json"

chunk_size = 10

df = raw_business_df.head(50)

llm_data = []

for i in range(1, len(df), chunk_size):
    chunk_df = df.iloc[i:i + chunk_size]

    llm_input = [
        {
            "id": row["id"],
            "name": row["name"],
            "address": row["address"],
            "categories": row["categories"]
        }
        for _, row in chunk_df.iterrows()
    ]

    llm_output = llm_clean_business(llm_input)
    llm_data.extend(llm_output)

# Append the Chunk Results to a JSON File
if os.path.exists(CLEAN_BUSINESS_JSON):
    os.remove(CLEAN_BUSINESS_JSON)

with open(CLEAN_BUSINESS_JSON, "w") as f:
    json.dump(llm_data, f, indent=2)

In [ ]:
from openai import OpenAI
import pandas as pd
import json
import math

client = OpenAI(api_key="YOUR_API_KEY")

# -------------------------
# LLM Helper Function
# -------------------------
def call_llm(batch):
    system = """You are a data cleaning assistant. 
For each business, return standardized fields using this JSON schema:
{
  "name": "",
  "street": "",
  "city": "",
  "state": "",
  "zip": "",
  "main_category": "",
  "sub_category": "",
  "micro_category": ""
}
Fix typos. Normalize names. Extract fields. Return VALID JSON ONLY.
"""

    user = f"Clean the following business records:\n{json.dumps(batch, indent=2)}"

    response = client.chat.completions.create(
        model="gpt-4.1",
        temperature=0,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )

    return json.loads(response.choices[0].message.content)

# -------------------------
# BATCH PROCESSING
# -------------------------
def batch_process(df, batch_size=150):
    cleaned_results = []

    for i in range(0, len(df), batch_size):
        chunk = df.iloc[i:i+batch_size]

        # Convert chunk to list of dicts for LLM
        llm_input = [
            {
                "name": row["business_name"],
                "address": row["address"],
                "lat": row["latitude"],
                "lon": row["longitude"]
            }
            for _, row in chunk.iterrows()
        ]

        print(f"Processing {i} to {i+batch_size}...")

        llm_output = call_llm(llm_input)

        # Merge LLM output + geo validation
        for (orig, cleaned) in zip(llm_input, llm_output):
            cleaned["invalid_geo"] = not validate_geo(orig["lat"], orig["lon"])
            cleaned_results.append(cleaned)

    return pd.DataFrame(cleaned_results)

# -------------------------
# RUN THE PIPELINE
# -------------------------
df = pd.read_csv("san_diego_businesses.csv")

cleaned_df = batch_process(df)

cleaned_df.to_csv("san_diego_businesses_cleaned.csv", index=False)
